# Notebook de Colab — Entrenamiento, Evaluación y XAI

**Tesis:** Sistema Inteligente de Moderación Automática en Redes Sociales para la Protección del Bienestar Digital del Usuario  
**Modelo central:** BETO (BERT en español) ajustado con corpus enriquecido con modismos latinoamericanos  

---

## ¿Qué hace este notebook?

Este notebook cubre íntegramente las fases del proyecto que requieren GPU. Ejecuta las celdas de **arriba hacia abajo**, una a una.

| Fase | Descripción | Salidas en Drive |
|------|-------------|------------------|
| **Fase 2** | Fine-tuning de BETO, mBERT y XLM-R (3 semillas c/u) + selección del mejor BETO | `models/` (10 carpetas) |
| **Fase 3** | Evaluación en test set, bootstrap (IC 95%), test de McNemar | `reports/tables/`, `reports/predictions/` |
| **Fase 4** | Segmentación por modismos LATAM, validación estadística de H3 | `reports/tables/h3_idiom_analysis/` |
| **Fase 5A** | SHAP sobre errores de BETO — análisis de explicabilidad | `reports/tables/xai_analysis/` |

---

## Antes de empezar: estructura en Google Drive

Sube desde tu PC local a `Mi unidad/unmsm/ciclo 2026-1/tesis/COLAB/`:

```
COLAB/
├── data/processed/
│   ├── train.parquet
│   ├── val.parquet
│   ├── test.parquet
│   └── corpus_v1_enriquecido.parquet
└── scripts/
    ├── train_model.py
    └── evaluate_model.py
```

**Configuración de Colab:**  
`Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **GPU T4**

---

## Al terminar: descargar a tu PC

Descarga desde Drive a `Tesis_Proyecto/`:

```
models/                              → Tesis_Proyecto/models/
reports/tables/                      → Tesis_Proyecto/reports/tables/
reports/predictions/                 → Tesis_Proyecto/reports/predictions/
reports/tables/xai_analysis/         → Tesis_Proyecto/reports/tables/xai_analysis/
```

> **Si Colab se desconecta:** los archivos ya guardados en Drive quedan intactos. Solo vuelve a montar Drive (celda 1) y continúa desde la celda siguiente.

---


In [1]:
print("hello")

hello


In [ ]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess

subprocess.run([
    "pip", "install", "--quiet",
    "transformers==4.47.1",
    "datasets",
    "scikit-learn",
    "pandas",
    "pyarrow",
    "accelerate"
], check=True)

import torch
import transformers

print("✅ GPU disponible:", torch.cuda.is_available())
print(
    "   Dispositivo:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)
print("   PyTorch:", torch.__version__)
print("   Transformers:", transformers.__version__)

Mounted at /content/drive
✅ GPU disponible: True
   Dispositivo: Tesla T4
   PyTorch: 2.11.0+cu128
   Transformers: 4.47.1


In [3]:
import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

print("✅ Imports correctos")
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.is_available())

✅ Imports correctos
Torch: 2.11.0+cu128
Transformers: 4.47.1
GPU: True


In [4]:
# ── Celda 2: Rutas ──────────────────────────────────────────────
import os
from pathlib import Path

# Ajusta esta ruta si usaste un nombre de carpeta diferente en Drive
DRIVE_ROOT = Path("/content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB")
DATA_DIR   = DRIVE_ROOT / "data" / "processed"
MODELS_DIR = DRIVE_ROOT / "models"
SCRIPT_PATH = DRIVE_ROOT / "scripts" / "train_model.py"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Verificar que los Parquets existen
for fname in ["train.parquet", "val.parquet", "test.parquet"]:
    fpath = DATA_DIR / fname
    if fpath.exists():
        import pandas as pd
        df = pd.read_parquet(fpath)
        print(f"✅ {fname}: {len(df)} filas")
    else:
        print(f"❌ NO ENCONTRADO: {fpath}  ← verifica la ruta en Drive")

✅ train.parquet: 23090 filas
✅ val.parquet: 4948 filas
✅ test.parquet: 4949 filas


In [ ]:
# ── Celda 3: Preparar script de entrenamiento ───────────────────
import shutil
from pathlib import Path

SOURCE_SCRIPT = SCRIPT_PATH
LOCAL_SCRIPT = Path("/content/train_model.py")

# Verificar que el script original exista
if not SOURCE_SCRIPT.exists():
    raise FileNotFoundError(
        f"No se encontró train_model.py en:\n{SOURCE_SCRIPT}"
    )

# Copiar el script desde Drive al almacenamiento temporal de Colab
shutil.copy2(SOURCE_SCRIPT, LOCAL_SCRIPT)

# Leer la copia temporal
content = LOCAL_SCRIPT.read_text(encoding="utf-8")

# Guardar el contenido original para verificar cambios
content_original = content

# Ajustar las rutas de los datos
content = content.replace(
    '"data/processed/train.parquet"',
    repr(str(DATA_DIR / "train.parquet"))
)

content = content.replace(
    '"data/processed/val.parquet"',
    repr(str(DATA_DIR / "val.parquet"))
)

content = content.replace(
    '"data/processed/test.parquet"',
    repr(str(DATA_DIR / "test.parquet"))
)

# Ajustar la carpeta de salida de los modelos
content = content.replace(
    'output_dir = f"models/{args.model}_finetuned_{args.seed}"',
    f'output_dir = f"{MODELS_DIR}/{{args.model}}_finetuned_{{args.seed}}"'
)

# Verificar que realmente se haya modificado el script
if content == content_original:
    raise RuntimeError(
        "No se realizó ningún reemplazo. "
        "Las líneas de train_model.py podrían tener un formato diferente."
    )

# Guardar la versión modificada solo en /content
LOCAL_SCRIPT.write_text(content, encoding="utf-8")

print("✅ Script copiado y rutas ajustadas")
print("   Script original :", SOURCE_SCRIPT)
print("   Script temporal :", LOCAL_SCRIPT)
print("   Datos           :", DATA_DIR)
print("   Modelos         :", MODELS_DIR)

✅ Script copiado y rutas ajustadas
   Script original : /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/scripts/train_model.py
   Script temporal : /content/train_model.py
   Datos           : /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/data/processed
   Modelos         : /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/models


In [ ]:
# Mostrar las líneas relevantes del script modificado
for numero, linea in enumerate(
    LOCAL_SCRIPT.read_text(encoding="utf-8").splitlines(),
    start=1
):
    if (
        "train.parquet" in linea
        or "val.parquet" in linea
        or "test.parquet" in linea
        or "output_dir" in linea
    ):
        print(f"{numero:03}: {linea}")

093:     train_df = pd.read_parquet('/content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/data/processed/train.parquet')
094:     val_df = pd.read_parquet('/content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/data/processed/val.parquet')
124:     output_dir = f"/content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/models/{args.model}_finetuned_{args.seed}"
126:         output_dir=output_dir,
160:     print(f"Guardando en {output_dir}...")
161:     trainer.save_model(output_dir)
162:     tokenizer.save_pretrained(output_dir)


In [ ]:
# ── Celda: BETO seed=42 ─────────────────────────────────────────
os.chdir("/content")
!python train_model.py --model beto --seed 42
print("🎉 BETO seed=42 completado")

 INICIO ENTRENAMIENTO
Modelo: beto
Semilla: 42
Dispositivo: cuda
PyTorch: 2.11.0+cu128
Transformers: 4.47.1
Cargando corpus...
Class weights: [0.64783121 2.19111786]
Tokenizando...
tokenizer_config.json: 100% 364/364 [00:00<00:00, 2.57MB/s]
config.json: 100% 648/648 [00:00<00:00, 4.57MB/s]
vocab.txt: 242kB [00:00, 22.1MB/s]
tokenizer.json: 480kB [00:00, 46.5MB/s]
special_tokens_map.json: 100% 134/134 [00:00<00:00, 1.21MB/s]
Map: 100% 23090/23090 [00:03<00:00, 6809.11 examples/s]
Map: 100% 4948/4948 [00:00<00:00, 8539.17 examples/s]
Cargando modelo...
pytorch_model.bin: 100% 440M/440M [00:03<00:00, 114MB/s]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Some weights of BertForSequenceClassification were not initial

In [ ]:
os.chdir("/content")

!python train_model.py --model beto --seed 123
print("🎉 BETO seed=123 completado")

 INICIO ENTRENAMIENTO
Modelo: beto
Semilla: 123
Dispositivo: cuda
PyTorch: 2.11.0+cu128
Transformers: 4.47.1
Cargando corpus...
Class weights: [0.64783121 2.19111786]
Tokenizando...
Map: 100% 23090/23090 [00:02<00:00, 8096.28 examples/s]
Map: 100% 4948/4948 [00:01<00:00, 4623.25 examples/s]
Cargando modelo...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference

In [ ]:
os.chdir("/content")

!python train_model.py --model beto --seed 2024
print("🎉 BETO seed=2024 completado")

 INICIO ENTRENAMIENTO
Modelo: beto
Semilla: 2024
Dispositivo: cuda
PyTorch: 2.11.0+cu128
Transformers: 4.47.1
Cargando corpus...
Class weights: [0.64783121 2.19111786]
Tokenizando...
Map: 100% 23090/23090 [00:03<00:00, 6113.01 examples/s]
Map: 100% 4948/4948 [00:01<00:00, 4843.94 examples/s]
Cargando modelo...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inferenc

In [ ]:
os.chdir("/content")

!python train_model.py --model mbert --seed 42
print("🎉 mBERT seed=42 completado")

 INICIO ENTRENAMIENTO
Modelo: mbert
Semilla: 42
Dispositivo: cuda
PyTorch: 2.11.0+cu128
Transformers: 4.47.1
Cargando corpus...
Class weights: [0.64783121 2.19111786]
Tokenizando...
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 356kB/s]
config.json: 100% 625/625 [00:00<00:00, 5.53MB/s]
vocab.txt: 996kB [00:00, 2.06MB/s]
tokenizer.json: 1.96MB [00:00, 2.88MB/s]
Map: 100% 23090/23090 [00:03<00:00, 6385.62 examples/s]
Map: 100% 4948/4948 [00:00<00:00, 8490.94 examples/s]
Cargando modelo...
model.safetensors: 100% 714M/714M [00:06<00:00, 115MB/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/train_model.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use

In [ ]:
os.chdir("/content")

!python train_model.py --model mbert --seed 123
print("🎉 mBERT seed=123 completado")

 INICIO ENTRENAMIENTO
Modelo: mbert
Semilla: 123
Dispositivo: cuda
PyTorch: 2.11.0+cu128
Transformers: 4.47.1
Cargando corpus...
Class weights: [0.64783121 2.19111786]
Tokenizando...
Map: 100% 23090/23090 [00:02<00:00, 8516.00 examples/s]
Map: 100% 4948/4948 [00:00<00:00, 8427.05 examples/s]
Cargando modelo...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/train_model.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
Iniciando entrenamiento...
{'loss': 0.7095, 'grad_norm': 4.265782833099365, 'learning_rate': 1.6262975778546713e-06, 'epoch': 0.03}
{'loss': 0.6992, 'grad_norm': 7.69218826293945

In [ ]:
os.chdir("/content")

!python train_model.py --model mbert --seed 2024
print("🎉 mBERT seed=2024 completado")

 INICIO ENTRENAMIENTO
Modelo: mbert
Semilla: 2024
Dispositivo: cuda
PyTorch: 2.11.0+cu128
Transformers: 4.47.1
Cargando corpus...
Class weights: [0.64783121 2.19111786]
Tokenizando...
Map: 100% 23090/23090 [00:02<00:00, 8028.89 examples/s]
Map: 100% 4948/4948 [00:00<00:00, 8133.39 examples/s]
Cargando modelo...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/train_model.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
Iniciando entrenamiento...
{'loss': 0.6918, 'grad_norm': 2.9472763538360596, 'learning_rate': 1.6608996539792389e-06, 'epoch': 0.03}
{'loss': 0.6925, 'grad_norm': 3.357720613479

In [ ]:
os.chdir("/content")

!python train_model.py --model xlmr --seed 42
print("🎉 XLM-R seed=42 completado")

 INICIO ENTRENAMIENTO
Modelo: xlmr
Semilla: 42
Dispositivo: cuda
PyTorch: 2.11.0+cu128
Transformers: 4.47.1
Cargando corpus...
Class weights: [0.64783121 2.19111786]
Tokenizando...
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 136kB/s]
config.json: 100% 615/615 [00:00<00:00, 5.39MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:01<00:00, 4.39MB/s]
tokenizer.json: 9.10MB [00:01, 7.76MB/s]
Map: 100% 23090/23090 [00:02<00:00, 7740.33 examples/s]
Map: 100% 4948/4948 [00:00<00:00, 7020.80 examples/s]
Cargando modelo...
model.safetensors: 100% 1.12G/1.12G [00:10<00:00, 109MB/s]
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/train_model.py:66: FutureWarning: `t

In [ ]:
os.chdir("/content")

!python train_model.py --model xlmr --seed 123
print("🎉 XLM-R seed=123 completado")

 INICIO ENTRENAMIENTO
Modelo: xlmr
Semilla: 123
Dispositivo: cuda
PyTorch: 2.11.0+cu128
Transformers: 4.47.1
Cargando corpus...
Class weights: [0.64783121 2.19111786]
Tokenizando...
Map: 100% 23090/23090 [00:03<00:00, 7604.36 examples/s]
Map: 100% 4948/4948 [00:00<00:00, 7809.44 examples/s]
Cargando modelo...
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/train_model.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
Iniciando entrenamiento...
{'loss': 0.7013, 'grad_norm': 1.9915271997451782, 'learning_rate': 8.477508650519031e-07

In [ ]:
os.chdir("/content")

!python train_model.py --model xlmr --seed 2024
print("🎉 XLM-R seed=2024 completado")

 INICIO ENTRENAMIENTO
Modelo: xlmr
Semilla: 2024
Dispositivo: cuda
PyTorch: 2.11.0+cu128
Transformers: 4.47.1
Cargando corpus...
Class weights: [0.64783121 2.19111786]
Tokenizando...
Map: 100% 23090/23090 [00:03<00:00, 6522.23 examples/s]
Map: 100% 4948/4948 [00:01<00:00, 4068.44 examples/s]
Cargando modelo...
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/train_model.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
Iniciando entrenamiento...
{'loss': 0.6959, 'grad_norm': 2.9037647247314453, 'learning_rate': 8.650519031141868e-0

In [ ]:
# ============================================================
# PARTE D — Seleccionar el mejor BETO (Paso 2.5)
# ============================================================

import json
import shutil
from pathlib import Path

# MODELS_DIR ya debe estar definido en la celda de rutas:
# MODELS_DIR = DRIVE_ROOT / "models"

SEEDS = [42, 123, 2024]
f1_scores = {}

print("Buscando modelos en:")
print(MODELS_DIR)
print()

for seed in SEEDS:
    model_dir = MODELS_DIR / f"beto_finetuned_{seed}"

    if not model_dir.exists():
        print(f"⚠️ No existe la carpeta: {model_dir.name}")
        continue

    # Buscar trainer_state.json dentro de los checkpoints
    state_files = list(
        model_dir.glob("checkpoint-*/trainer_state.json")
    )

    if not state_files:
        print(
            f"⚠️ No se encontró trainer_state.json "
            f"para BETO seed={seed}"
        )
        continue

    # Seleccionar el trainer_state del checkpoint más reciente
    state_file = max(
        state_files,
        key=lambda path: int(path.parent.name.split("-")[-1])
    )

    state = json.loads(
        state_file.read_text(encoding="utf-8")
    )

    best_f1 = state.get("best_metric")
    best_checkpoint = state.get("best_model_checkpoint")

    if best_f1 is None:
        print(
            f"⚠️ No se encontró best_metric "
            f"para BETO seed={seed}"
        )
        continue

    f1_scores[seed] = float(best_f1)

    print(f"BETO seed {seed}")
    print(f"  F1 validación: {best_f1:.4f}")
    print(f"  Mejor checkpoint: {best_checkpoint}")
    print()


# Verificar que estén las tres corridas
if len(f1_scores) != len(SEEDS):
    print(
        f"❌ Solo se encontraron {len(f1_scores)} "
        f"de las {len(SEEDS)} corridas de BETO."
    )
    print("No se creó todavía beto_finetuned_final.")

else:
    # Seleccionar la semilla con mayor F1 de validación
    best_seed = max(f1_scores, key=f1_scores.get)
    best_f1 = f1_scores[best_seed]

    print("=" * 55)
    print(f"🏆 Mejor semilla BETO: {best_seed}")
    print(f"🏆 Mejor F1 de validación: {best_f1:.4f}")
    print("=" * 55)

    # La fuente es la carpeta raíz del entrenamiento,
    # no uno de los checkpoints
    src = MODELS_DIR / f"beto_finetuned_{best_seed}"
    dst = MODELS_DIR / "beto_finetuned_final"

    # Eliminar una selección anterior, si existe
    if dst.exists():
        shutil.rmtree(dst)

    # Copiar el modelo final de la raíz,
    # excluyendo los checkpoints para evitar duplicar espacio
    shutil.copytree(
        src,
        dst,
        ignore=shutil.ignore_patterns("checkpoint-*")
    )

    print()
    print("✅ Modelo BETO final copiado correctamente:")
    print(dst)

    print()
    print("Archivos del modelo final:")
    for file in sorted(dst.iterdir()):
        print(f"  - {file.name}")

Buscando modelos en:
/content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/models

BETO seed 42
  F1 validación: 0.7186
  Mejor checkpoint: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/models/beto_finetuned_42/checkpoint-2888

BETO seed 123
  F1 validación: 0.7049
  Mejor checkpoint: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/models/beto_finetuned_123/checkpoint-5776

BETO seed 2024
  F1 validación: 0.7019
  Mejor checkpoint: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/models/beto_finetuned_2024/checkpoint-2888

🏆 Mejor semilla BETO: 42
🏆 Mejor F1 de validación: 0.7186

✅ Modelo BETO final copiado correctamente:
/content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/models/beto_finetuned_final

Archivos del modelo final:
  - config.json
  - model.safetensors
  - special_tokens_map.json
  - tokenizer.json
  - tokenizer_config.json
  - training_args.bin
  - vocab.txt


In [6]:
# ── Celda: Fase 3.1 y 3.2 - Evaluación en test set ─────────────

from pathlib import Path
import shutil
import subprocess
import sys

# Estas rutas deben existir desde la celda 2:
# DRIVE_ROOT, DATA_DIR y MODELS_DIR

REPORTS_DIR = DRIVE_ROOT / "reports" / "tables"
PREDS_DIR = DRIVE_ROOT / "reports" / "predictions"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)

eval_script_drive = DRIVE_ROOT / "scripts" / "evaluate_model.py"
eval_script_tmp = Path("/content/evaluate_model.py")
test_path = DATA_DIR / "test.parquet"

print("📁 Rutas utilizadas:")
print(f"   Test: {test_path}")
print(f"   Modelos: {MODELS_DIR}")
print(f"   Reportes: {REPORTS_DIR}")
print(f"   Predicciones: {PREDS_DIR}")

# Verificar archivos necesarios
required_paths = [
    test_path,
    MODELS_DIR,
    eval_script_drive,
]

missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    print("\n❌ Faltan archivos o carpetas:")
    for path in missing_paths:
        print(f"   - {path}")

else:
    # Copiar el script desde Drive hacia /content
    shutil.copy2(eval_script_drive, eval_script_tmp)
    print(f"\n✅ Script copiado a: {eval_script_tmp}")

    # Leer el script temporal
    script_content = eval_script_tmp.read_text(encoding="utf-8")

    # Ajustar ruta del test
    script_content = script_content.replace(
        'Path("data/processed/test.parquet")',
        f'Path(r"{test_path}")'
    )

    # Ajustar carpeta de predicciones
    script_content = script_content.replace(
        'Path("reports/predictions")',
        f'Path(r"{PREDS_DIR}")'
    )

    # Ajustar carpeta de tablas
    script_content = script_content.replace(
        'Path("reports/tables")',
        f'Path(r"{REPORTS_DIR}")'
    )

    # Ajustar ruta de los modelos
    script_content = script_content.replace(
        'Path(f"models/{model_name}_finetuned_{seed}")',
        f'Path(r"{MODELS_DIR}") / f"{{model_name}}_finetuned_{{seed}}"'
    )

    # Guardar el script modificado
    eval_script_tmp.write_text(
        script_content,
        encoding="utf-8"
    )

    print("✅ Rutas ajustadas correctamente")
    print("\n🚀 Ejecutando evaluación...\n")

    result = subprocess.run(
        [sys.executable, str(eval_script_tmp), "--all"]
    )

    if result.returncode == 0:
        print("\n✅ Evaluación completada correctamente")
        print(f"📊 Tablas: {REPORTS_DIR}")
        print(f"📝 Predicciones: {PREDS_DIR}")
    else:
        print(
            f"\n❌ La evaluación terminó con código "
            f"{result.returncode}"
        )

📁 Rutas utilizadas:
   Test: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/data/processed/test.parquet
   Modelos: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/models
   Reportes: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables
   Predicciones: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/predictions

✅ Script copiado a: /content/evaluate_model.py
✅ Rutas ajustadas correctamente

🚀 Ejecutando evaluación...


✅ Evaluación completada correctamente
📊 Tablas: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables
📝 Predicciones: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/predictions


In [7]:
# ── Celda: Paso 3.3 - Bootstrap e intervalos de confianza ──────────────

import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

def bootstrap_ic(y_true, y_pred, B=1000, alpha=0.05, seed=42):
    """Calcula intervalo de confianza por bootstrap."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        vals[b] = f1_score(y_true[idx], y_pred[idx], average="binary", pos_label=1, zero_division=0)
    lo = np.percentile(vals, 100 * alpha / 2)
    hi = np.percentile(vals, 100 * (1 - alpha / 2))
    return vals.mean(), lo, hi

# Cargar test set
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
y_true = test_df["etiqueta"].values

print("="*70)
print("  PASO 3.3 - BOOTSTRAP E INTERVALOS DE CONFIANZA (95%)")
print("="*70)

results_bootstrap = []

for model_name in ["beto", "mbert", "xlmr"]:
    print(f"\n{model_name.upper()}:")
    for seed in [42, 123, 2024]:
        # Cargar predicciones del CSV generado en 3.2
        pred_file = PREDS_DIR / f"{model_name}_{seed}_preds.csv"
        if not pred_file.exists():
            print(f"  ❌ {pred_file} no encontrado")
            continue

        preds_df = pd.read_csv(pred_file)
        y_pred = preds_df["y_pred"].values

        # Bootstrap
        f1_mean, f1_lo, f1_hi = bootstrap_ic(y_true, y_pred, B=1000)

        print(f"  Semilla {seed}: F1 = {f1_mean:.4f}  [{f1_lo:.4f}, {f1_hi:.4f}]")

        results_bootstrap.append({
            "model": model_name,
            "seed": seed,
            "f1_mean": f1_mean,
            "f1_lo": f1_lo,
            "f1_hi": f1_hi,
        })

# Resumen por modelo
print(f"\n{'='*70}")
print("  RESUMEN BOOTSTRAP POR MODELO")
print(f"{'='*70}")

for model_name in ["beto", "mbert", "xlmr"]:
    sub = [r for r in results_bootstrap if r["model"] == model_name]
    if sub:
        f1s = [r["f1_mean"] for r in sub]
        f1_mean = np.mean(f1s)
        f1_std = np.std(f1s)
        print(f"{model_name.upper():6s}: F1 = {f1_mean:.4f} ± {f1_std:.4f}")

# Guardar resultados
df_bootstrap = pd.DataFrame(results_bootstrap)
bootstrap_file = REPORTS_DIR / "bootstrap_ic.csv"
df_bootstrap.to_csv(bootstrap_file, index=False)
print(f"\n✅ Resultados guardados en {bootstrap_file}")

  PASO 3.3 - BOOTSTRAP E INTERVALOS DE CONFIANZA (95%)

BETO:
  Semilla 42: F1 = 0.6817  [0.6595, 0.7026]
  Semilla 123: F1 = 0.6862  [0.6628, 0.7083]
  Semilla 2024: F1 = 0.6761  [0.6531, 0.6989]

MBERT:
  Semilla 42: F1 = 0.6519  [0.6293, 0.6724]
  Semilla 123: F1 = 0.6505  [0.6275, 0.6710]
  Semilla 2024: F1 = 0.6412  [0.6163, 0.6651]

XLMR:
  Semilla 42: F1 = 0.6722  [0.6493, 0.6949]
  Semilla 123: F1 = 0.6612  [0.6378, 0.6825]
  Semilla 2024: F1 = 0.6723  [0.6507, 0.6934]

  RESUMEN BOOTSTRAP POR MODELO
BETO  : F1 = 0.6813 ± 0.0041
MBERT : F1 = 0.6479 ± 0.0048
XLMR  : F1 = 0.6686 ± 0.0052

✅ Resultados guardados en /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables/bootstrap_ic.csv


In [9]:
# ── Celda: Paso 3.4 - Test de McNemar ────────────────────────────────

from statsmodels.stats.contingency_tables import mcnemar

print("="*70)
print("  PASO 3.4 - TEST DE MCNEMAR (Significancia estadística)")
print("="*70)

results_mcnemar = []

# Comparaciones principales: BETO vs mBERT, BETO vs XLM-R
comparisons = [
    ("beto", "mbert"),
    ("beto", "xlmr"),
    ("mbert", "xlmr"),
]

# Usar la mejor semilla (42 por defecto, o ajusta según necesites)
best_seed = 42

for model1, model2 in comparisons:
    # Cargar predicciones
    pred1_file = PREDS_DIR / f"{model1}_{best_seed}_preds.csv"
    pred2_file = PREDS_DIR / f"{model2}_{best_seed}_preds.csv"

    if not pred1_file.exists() or not pred2_file.exists():
        print(f"  ❌ {model1} o {model2} no encontrados")
        continue

    preds1_df = pd.read_csv(pred1_file)
    preds2_df = pd.read_csv(pred2_file)

    y_pred1 = preds1_df["y_pred"].values
    y_pred2 = preds2_df["y_pred"].values

    # Construir tabla 2x2 de aciertos/errores
    aciertos1 = (y_pred1 == y_true).astype(int)
    aciertos2 = (y_pred2 == y_true).astype(int)

    n00 = ((aciertos1 == 1) & (aciertos2 == 1)).sum()  # Ambos acierto
    n01 = ((aciertos1 == 1) & (aciertos2 == 0)).sum()  # Solo model1 acierto
    n10 = ((aciertos1 == 0) & (aciertos2 == 1)).sum()  # Solo model2 acierto
    n11 = ((aciertos1 == 0) & (aciertos2 == 0)).sum()  # Ambos error

    tabla = [[n00, n01], [n10, n11]]

    # McNemar test
    res = mcnemar(tabla, exact=False, correction=True)

    is_sig = "✅ SÍ" if res.pvalue < 0.05 else "❌ NO"

    print(f"\n{model1.upper()} vs {model2.upper()} (semilla {best_seed}):")
    print(f"  Tabla 2x2:")
    print(f"    Ambos acierto    : {n00}")
    print(f"    Solo {model1} acierto: {n01}")
    print(f"    Solo {model2} acierto: {n10}")
    print(f"    Ambos error      : {n11}")
    print(f"  McNemar p-valor : {res.pvalue:.6f}")
    print(f"  Significativo (α=0.05): {is_sig}")

    results_mcnemar.append({
        "model1": model1,
        "model2": model2,
        "seed": best_seed,
        "n00": n00,
        "n01": n01,
        "n10": n10,
        "n11": n11,
        "pvalue": res.pvalue,
        "significant": res.pvalue < 0.05,
    })

# Guardar resultados
df_mcnemar = pd.DataFrame(results_mcnemar)
mcnemar_file = REPORTS_DIR / "mcnemar_results.csv"
df_mcnemar.to_csv(mcnemar_file, index=False)
print(f"\n✅ Resultados guardados en {mcnemar_file}")

  PASO 3.4 - TEST DE MCNEMAR (Significancia estadística)

BETO vs MBERT (semilla 42):
  Tabla 2x2:
    Ambos acierto    : 3756
    Solo beto acierto: 440
    Solo mbert acierto: 261
    Ambos error      : 492
  McNemar p-valor : 0.000000
  Significativo (α=0.05): ✅ SÍ

BETO vs XLMR (semilla 42):
  Tabla 2x2:
    Ambos acierto    : 3882
    Solo beto acierto: 314
    Solo xlmr acierto: 275
    Ambos error      : 478
  McNemar p-valor : 0.117404
  Significativo (α=0.05): ❌ NO

MBERT vs XLMR (semilla 42):
  Tabla 2x2:
    Ambos acierto    : 3727
    Solo mbert acierto: 290
    Solo xlmr acierto: 430
    Ambos error      : 502
  McNemar p-valor : 0.000000
  Significativo (α=0.05): ✅ SÍ

✅ Resultados guardados en /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables/mcnemar_results.csv


In [15]:
# ── Celda: Paso 4.1 - Segmentación del test set ──────────────────────

import pandas as pd
import numpy as np

print("="*70)
print("  PASO 4.1 - SEGMENTACIÓN DEL TEST SET POR MODISMOS")
print("="*70)

# Crear directorio para resultados
H3_ANALYSIS_DIR = REPORTS_DIR / "h3_idiom_analysis"
H3_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# Cargar test set
test_df = pd.read_parquet(DATA_DIR / "test.parquet")

# Segmentar por modismos
test_with_idioms = test_df[test_df["tiene_modismo"] == True]
test_without_idioms = test_df[test_df["tiene_modismo"] == False]

print(f"\nTest set total: {len(test_df)} textos")
print(f"  ├─ CON modismos    : {len(test_with_idioms)} ({len(test_with_idioms)/len(test_df):.1%})")
print(f"  └─ SIN modismos    : {len(test_without_idioms)} ({len(test_without_idioms)/len(test_df):.1%})")

print(f"\n--- Distribución de clases ---")
print(f"\nCON modismos:")
print(test_with_idioms["etiqueta"].value_counts().sort_index())
print(f"  Hate: {(test_with_idioms['etiqueta']==1).sum()} ({(test_with_idioms['etiqueta']==1).sum()/len(test_with_idioms):.1%})")

print(f"\nSIN modismos:")
print(test_without_idioms["etiqueta"].value_counts().sort_index())
print(f"  Hate: {(test_without_idioms['etiqueta']==1).sum()} ({(test_without_idioms['etiqueta']==1).sum()/len(test_without_idioms):.1%})")

# Guardar estadísticas de segmentación
segmentation_stats = pd.DataFrame({
    "subset": ["con_modismos", "sin_modismos"],
    "n_total": [len(test_with_idioms), len(test_without_idioms)],
    "n_hate": [(test_with_idioms["etiqueta"]==1).sum(), (test_without_idioms["etiqueta"]==1).sum()],
    "n_no_hate": [(test_with_idioms["etiqueta"]==0).sum(), (test_without_idioms["etiqueta"]==0).sum()],
    "pct_hate": [
        (test_with_idioms["etiqueta"]==1).sum() / len(test_with_idioms),
        (test_without_idioms["etiqueta"]==1).sum() / len(test_without_idioms)
    ],
})

seg_file = H3_ANALYSIS_DIR / "h3_test_segmentation.csv"
segmentation_stats.to_csv(seg_file, index=False)
print(f"\n✅ Segmentación guardada en: {seg_file}")

  PASO 4.1 - SEGMENTACIÓN DEL TEST SET POR MODISMOS

Test set total: 4949 textos
  ├─ CON modismos    : 2686 (54.3%)
  └─ SIN modismos    : 2263 (45.7%)

--- Distribución de clases ---

CON modismos:
etiqueta
0    1838
1     848
Name: count, dtype: int64
  Hate: 848 (31.6%)

SIN modismos:
etiqueta
0    1982
1     281
Name: count, dtype: int64
  Hate: 281 (12.4%)

✅ Segmentación guardada en: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables/h3_idiom_analysis/h3_test_segmentation.csv


In [16]:
# ── Celda: Paso 4.2 - Evaluación de BETO en subconjuntos ──────────────

import torch
from transformers import pipeline
from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score, roc_auc_score,
    confusion_matrix
)

print("\n" + "="*70)
print("  PASO 4.2 - EVALUACIÓN DE BETO EN SUBCONJUNTOS")
print("="*70)

# Cargar modelo BETO final
model_path = MODELS_DIR / "beto_finetuned_final"
print(f"\nCargando modelo: {model_path}")

pipe = pipeline(
    "text-classification",
    model=str(model_path),
    device=0 if torch.cuda.is_available() else -1
)

# Función para evaluar en subset
def evaluate_subset(subset_df, subset_name):
    print(f"\n--- Evaluando subconjunto: {subset_name} ---")
    print(f"Textos: {len(subset_df)}")

    y_true = subset_df["etiqueta"].values
    y_pred_list = []
    y_proba_list = []

    # Inferencia
    for i, text in enumerate(subset_df["texto"]):
        if (i + 1) % 500 == 0:
            print(f"  Procesados {i+1}/{len(subset_df)}")

        output = pipe(text, truncation=True, max_length=128)
        label = 1 if output[0]["label"] == "LABEL_1" else 0
        prob = output[0]["score"]

        y_pred_list.append(label)
        y_proba_list.append(prob if output[0]["label"] == "LABEL_1" else 1 - prob)

    y_pred = np.array(y_pred_list)
    y_proba = np.array(y_proba_list)

    # Métricas
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_proba)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\n  Resultados:")
    print(f"    Precision (hate): {p:.4f}")
    print(f"    Recall (hate)   : {r:.4f}")
    print(f"    F1 (hate)       : {f:.4f}")
    print(f"    Accuracy        : {acc:.4f}")
    print(f"    ROC-AUC         : {roc_auc:.4f}")
    print(f"    Confusion Matrix: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

    return {
        "subset": subset_name,
        "n_samples": len(subset_df),
        "n_hate": (y_true == 1).sum(),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f),
        "accuracy": float(acc),
        "roc_auc": float(roc_auc),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }, y_true, y_pred, y_proba

# Evaluar ambos subconjuntos
results_subsets = []
data_for_h3_test = {}

for subset_name, subset_df in [
    ("con_modismos", test_with_idioms),
    ("sin_modismos", test_without_idioms)
]:
    result, y_true, y_pred, y_proba = evaluate_subset(subset_df, subset_name)
    results_subsets.append(result)
    data_for_h3_test[subset_name] = {
        "y_true": y_true,
        "y_pred": y_pred,
        "y_proba": y_proba
    }

# Guardar resultados
df_eval = pd.DataFrame(results_subsets)
eval_file = H3_ANALYSIS_DIR / "h3_beto_evaluation_subsets.csv"
df_eval.to_csv(eval_file, index=False)
print(f"\n✅ Evaluación guardada en: {eval_file}")

Device set to use cuda:0



  PASO 4.2 - EVALUACIÓN DE BETO EN SUBCONJUNTOS

Cargando modelo: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/models/beto_finetuned_final

--- Evaluando subconjunto: con_modismos ---
Textos: 2686
  Procesados 500/2686
  Procesados 1000/2686
  Procesados 1500/2686
  Procesados 2000/2686
  Procesados 2500/2686

  Resultados:
    Precision (hate): 0.6902
    Recall (hate)   : 0.7854
    F1 (hate)       : 0.7347
    Accuracy        : 0.8209
    ROC-AUC         : 0.8968
    Confusion Matrix: TN=1539, FP=299, FN=182, TP=666

--- Evaluando subconjunto: sin_modismos ---
Textos: 2263
  Procesados 500/2263
  Procesados 1000/2263
  Procesados 1500/2263
  Procesados 2000/2263

  Resultados:
    Precision (hate): 0.5167
    Recall (hate)   : 0.4947
    F1 (hate)       : 0.5055
    Accuracy        : 0.8798
    ROC-AUC         : 0.8523
    Confusion Matrix: TN=1852, FP=130, FN=142, TP=139

✅ Evaluación guardada en: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables/h3_idi

In [17]:
# ── Celda: Paso 4.3 - Validación estadística de H3 ──────────────────

from scipy import stats
from sklearn.metrics import f1_score

print("\n" + "="*70)
print("  PASO 4.3 - VALIDACIÓN ESTADÍSTICA DE H3")
print("="*70)

# Cargar datos de evaluación
df_eval = pd.read_csv(H3_ANALYSIS_DIR / "h3_beto_evaluation_subsets.csv")

f1_con = df_eval[df_eval["subset"] == "con_modismos"]["f1"].values[0]
f1_sin = df_eval[df_eval["subset"] == "sin_modismos"]["f1"].values[0]
delta_f1_observed = f1_con - f1_sin

print(f"\nF1 (CON modismos)  : {f1_con:.4f}")
print(f"F1 (SIN modismos)  : {f1_sin:.4f}")
print(f"Δ F1 (observado)   : {delta_f1_observed:.4f}")

# Bootstrap sobre Δ F1
print(f"\n--- Bootstrap (B=1000) ---")

def bootstrap_delta_f1(y_true_con, y_pred_con, y_true_sin, y_pred_sin, B=1000, seed=42):
    """Bootstrap sobre la diferencia de F1"""
    rng = np.random.default_rng(seed)
    deltas = []

    n_con = len(y_true_con)
    n_sin = len(y_true_sin)

    for b in range(B):
        # Remuestreo con reemplazo
        idx_con = rng.choice(n_con, size=n_con, replace=True)
        idx_sin = rng.choice(n_sin, size=n_sin, replace=True)

        f1_b_con = f1_score(y_true_con[idx_con], y_pred_con[idx_con],
                             average="binary", pos_label=1, zero_division=0)
        f1_b_sin = f1_score(y_true_sin[idx_sin], y_pred_sin[idx_sin],
                             average="binary", pos_label=1, zero_division=0)

        deltas.append(f1_b_con - f1_b_sin)

    deltas = np.array(deltas)
    return deltas

deltas = bootstrap_delta_f1(
    data_for_h3_test["con_modismos"]["y_true"],
    data_for_h3_test["con_modismos"]["y_pred"],
    data_for_h3_test["sin_modismos"]["y_true"],
    data_for_h3_test["sin_modismos"]["y_pred"],
    B=1000
)

delta_mean = deltas.mean()
delta_std = deltas.std()
delta_ci_lo = np.percentile(deltas, 2.5)
delta_ci_hi = np.percentile(deltas, 97.5)

print(f"Δ F1 (media bootstrap) : {delta_mean:.4f}")
print(f"Δ F1 (std)             : {delta_std:.4f}")
print(f"IC 95% Δ F1            : [{delta_ci_lo:.4f}, {delta_ci_hi:.4f}]")

# Test de permutación (H0: no hay diferencia)
print(f"\n--- Test de permutación (H0: Δ F1 = 0) ---")

def permutation_test(y_true_con, y_pred_con, y_true_sin, y_pred_sin, n_perm=1000, seed=42):
    """Test de permutación para la diferencia de F1"""
    rng = np.random.default_rng(seed)

    # F1 observado
    f1_con_obs = f1_score(y_true_con, y_pred_con, average="binary", pos_label=1, zero_division=0)
    f1_sin_obs = f1_score(y_true_sin, y_pred_sin, average="binary", pos_label=1, zero_division=0)
    delta_obs = f1_con_obs - f1_sin_obs

    # Combinar predicciones
    y_pred_combined = np.concatenate([y_pred_con, y_pred_sin])

    # Permutar
    count_extreme = 0
    for _ in range(n_perm):
        perm_idx = rng.permutation(len(y_pred_combined))
        y_pred_perm = y_pred_combined[perm_idx]

        y_pred_con_perm = y_pred_perm[:len(y_pred_con)]
        y_pred_sin_perm = y_pred_perm[len(y_pred_con):]

        f1_con_perm = f1_score(y_true_con, y_pred_con_perm, average="binary", pos_label=1, zero_division=0)
        f1_sin_perm = f1_score(y_true_sin, y_pred_sin_perm, average="binary", pos_label=1, zero_division=0)
        delta_perm = f1_con_perm - f1_sin_perm

        if abs(delta_perm) >= abs(delta_obs):
            count_extreme += 1

    p_value = (count_extreme + 1) / (n_perm + 1)
    return p_value

p_value = permutation_test(
    data_for_h3_test["con_modismos"]["y_true"],
    data_for_h3_test["con_modismos"]["y_pred"],
    data_for_h3_test["sin_modismos"]["y_true"],
    data_for_h3_test["sin_modismos"]["y_pred"],
    n_perm=1000
)

print(f"p-valor (test permutación) : {p_value:.6f}")
print(f"Significativo (α=0.05)     : {'SÍ ✅' if p_value < 0.05 else 'NO ❌'}")

# Conclusión
print(f"\n{'='*70}")
print("  CONCLUSIÓN DE H3")
print(f"{'='*70}")

is_supported = delta_ci_lo > 0  # IC no cruza 0

if is_supported:
    print(f"\n✅ HIPÓTESIS H3 SOPORTADA")
    print(f"\n   BETO rinde {delta_f1_observed:.1%} MEJOR en textos CON modismos")
    print(f"   que en textos SIN modismos.")
    print(f"\n   Evidencia:")
    print(f"   • Δ F1 observado: {delta_f1_observed:.4f}")
    print(f"   • IC 95% (bootstrap): [{delta_ci_lo:.4f}, {delta_ci_hi:.4f}]")
    print(f"   • p-valor (permutación): {p_value:.6f}")
    print(f"   • Conclusión: El modelo CAPTURA mejor el contexto")
    print(f"                 cultural de modismos latinoamericanos.")
else:
    print(f"\n❌ HIPÓTESIS H3 NO SOPORTADA")
    print(f"\n   BETO NO demuestra mejora en textos CON modismos.")
    print(f"\n   Evidencia:")
    print(f"   • Δ F1 observado: {delta_f1_observed:.4f}")
    print(f"   • IC 95% (bootstrap): [{delta_ci_lo:.4f}, {delta_ci_hi:.4f}]")
    print(f"   • p-valor (permutación): {p_value:.6f}")
    print(f"   • Conclusión: No hay diferencia estadística significativa.")

# Guardar resultados formales
validation_results = pd.DataFrame({
    "metric": [
        "f1_con_modismos",
        "f1_sin_modismos",
        "delta_f1_observed",
        "delta_f1_bootstrap_mean",
        "delta_f1_bootstrap_std",
        "delta_f1_ci_lower",
        "delta_f1_ci_upper",
        "p_value_permutation",
        "significant_alpha_005",
        "h3_supported"
    ],
    "value": [
        f1_con,
        f1_sin,
        delta_f1_observed,
        delta_mean,
        delta_std,
        delta_ci_lo,
        delta_ci_hi,
        p_value,
        p_value < 0.05,
        is_supported
    ]
})

val_file = H3_ANALYSIS_DIR / "h3_hypothesis_validation.csv"
validation_results.to_csv(val_file, index=False)
print(f"\n✅ Validación formal guardada en: {val_file}")


  PASO 4.3 - VALIDACIÓN ESTADÍSTICA DE H3

F1 (CON modismos)  : 0.7347
F1 (SIN modismos)  : 0.5055
Δ F1 (observado)   : 0.2292

--- Bootstrap (B=1000) ---
Δ F1 (media bootstrap) : 0.2300
Δ F1 (std)             : 0.0290
IC 95% Δ F1            : [0.1738, 0.2889]

--- Test de permutación (H0: Δ F1 = 0) ---
p-valor (test permutación) : 0.000999
Significativo (α=0.05)     : SÍ ✅

  CONCLUSIÓN DE H3

✅ HIPÓTESIS H3 SOPORTADA

   BETO rinde 22.9% MEJOR en textos CON modismos
   que en textos SIN modismos.

   Evidencia:
   • Δ F1 observado: 0.2292
   • IC 95% (bootstrap): [0.1738, 0.2889]
   • p-valor (permutación): 0.000999
   • Conclusión: El modelo CAPTURA mejor el contexto
                 cultural de modismos latinoamericanos.

✅ Validación formal guardada en: /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables/h3_idiom_analysis/h3_hypothesis_validation.csv


In [19]:
# ── Celda: Paso 5.1 - Identificar errores de BETO ──────────────────

import pandas as pd
import numpy as np
from pathlib import Path

print("="*70)
print("  PASO 5.1 - IDENTIFICAR PREDICCIONES INCORRECTAS DE BETO")
print("="*70)

# Crear directorio para resultados XAI
XAI_DIR = REPORTS_DIR / "xai_analysis"
XAI_DIR.mkdir(parents=True, exist_ok=True)

# Cargar test set y predicciones del mejor BETO
test_df = pd.read_parquet(DATA_DIR / "test.parquet")

# Usa la semilla que tuvo mejor F1 en validación (ajustar si es diferente)
BEST_SEED = 42
preds_df = pd.read_csv(PREDS_DIR / f"beto_{BEST_SEED}_preds.csv")

# Combinar
test_df = test_df.reset_index(drop=True)
test_df["y_pred"] = preds_df["y_pred"]
test_df["prob_hate"] = preds_df["prob_hate"]

# Identificar errores
falsos_positivos = test_df[(test_df["etiqueta"] == 0) & (test_df["y_pred"] == 1)]
falsos_negativos = test_df[(test_df["etiqueta"] == 1) & (test_df["y_pred"] == 0)]

print(f"\nFalsos positivos (no-hate predicho como hate) : {len(falsos_positivos)}")
print(f"Falsos negativos (hate predicho como no-hate) : {len(falsos_negativos)}")

# Seleccionar 10 de cada tipo
fp_sample = falsos_positivos.sample(min(10, len(falsos_positivos)), random_state=42)
fn_sample = falsos_negativos.sample(min(10, len(falsos_negativos)), random_state=42)

wrong_preds = pd.concat([fp_sample, fn_sample])
wrong_preds["error_type"] = (
    ["falso_positivo"] * len(fp_sample) + ["falso_negativo"] * len(fn_sample)
)

# Guardar
wrong_file = XAI_DIR / "shap_wrong_predictions.csv"
wrong_preds[["texto", "etiqueta", "y_pred", "prob_hate", "tiene_modismo", "error_type"]].to_csv(
    wrong_file, index=False
)
print(f"\n✅ {len(wrong_preds)} errores seleccionados → {wrong_file}")

  PASO 5.1 - IDENTIFICAR PREDICCIONES INCORRECTAS DE BETO

Falsos positivos (no-hate predicho como hate) : 429
Falsos negativos (hate predicho como no-hate) : 324

✅ 20 errores seleccionados → /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables/xai_analysis/shap_wrong_predictions.csv


In [20]:
# ── Celda: Paso 5.2 - Generar explicaciones SHAP ───────────────────

import shap
import torch
import json
from transformers import AutoTokenizer, pipeline

print("="*70)
print("  PASO 5.2 - EXPLICACIONES SHAP SOBRE ERRORES DE BETO")
print("="*70)

# Cargar modelo y tokenizer
model_path = str(MODELS_DIR / "beto_finetuned_final")
tokenizer = AutoTokenizer.from_pretrained(model_path)

pipe = pipeline(
    "text-classification",
    model=model_path,
    tokenizer=tokenizer,
    top_k=None,
    device=0 if torch.cuda.is_available() else -1,
)

# Construir explainer SHAP
masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(pipe, masker)

# Cargar errores seleccionados
wrong_preds = pd.read_csv(XAI_DIR / "shap_wrong_predictions.csv")
textos = wrong_preds["texto"].tolist()

print(f"\nGenerando explicaciones para {len(textos)} textos...")
print("(puede tardar 20-30 minutos en GPU T4)\n")

resultados_shap = []

for i, row in wrong_preds.iterrows():
    texto = row["texto"]
    print(f"  [{wrong_preds.index.get_loc(i)+1}/{len(wrong_preds)}] {row['error_type']} — {texto[:60]}...")

    shap_values = explainer([texto])

    # Extraer tokens y pesos para la clase hate (LABEL_1)
    tokens = shap_values.data[0]
    pesos  = shap_values.values[0][:, 1].tolist()  # columna 1 = hate

    # Top 5 tokens por peso absoluto
    token_peso = sorted(zip(tokens, pesos), key=lambda x: abs(x[1]), reverse=True)
    top5 = token_peso[:5]

    print(f"    Top tokens: {[(t, round(p,3)) for t,p in top5]}")

    resultados_shap.append({
        "texto":          texto,
        "etiqueta":       int(row["etiqueta"]),
        "y_pred":         int(row["y_pred"]),
        "error_type":     row["error_type"],
        "tiene_modismo":  row["tiene_modismo"],
        "tokens":         list(tokens),
        "pesos":          pesos,
        "top5_tokens":    json.dumps([t for t,_ in top5]),
        "top5_pesos":     json.dumps([round(p,4) for _,p in top5]),
    })

# Guardar resultados
shap_file = XAI_DIR / "shap_analysis_results.csv"
df_shap = pd.DataFrame(resultados_shap)
df_shap[["texto", "etiqueta", "y_pred", "error_type", "tiene_modismo",
         "top5_tokens", "top5_pesos"]].to_csv(shap_file, index=False)

# Guardar JSON completo (tokens + pesos de todos)
json_file = XAI_DIR / "shap_full_weights.json"
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(resultados_shap, f, ensure_ascii=False, indent=2)

print(f"\n✅ Análisis SHAP guardado:")
print(f"   CSV  → {shap_file}")
print(f"   JSON → {json_file}")

  PASO 5.2 - EXPLICACIONES SHAP SOBRE ERRORES DE BETO


Device set to use cuda:0



Generando explicaciones para 20 textos...
(puede tardar 20-30 minutos en GPU T4)

  [1/20] falso_positivo — @USER Uy, por quién me tomas, por una loca agresiva? 😅😅...
    Top tokens: [('loca ', 0.529), ('agresiva', 0.189), (', ', 0.104), ('me ', -0.062), ('una ', 0.049)]
  [2/20] falso_positivo — @USER Venga, todos los fachas conmigo...EA EA EA, VENEZUELA ...


  0%|          | 0/498 [00:00<?, ?it/s]

    Top tokens: [('fac', 0.132), ('has ', 0.109), ('Venga', 0.062), ('USE', 0.06), ('VE', 0.049)]
  [3/20] falso_positivo — @USER @USER Los fascistas cuando son imbéciles, se denominan...
    Top tokens: [('imbéciles', 0.177), ('fas', 0.131), (', ', 0.068), ('se ', 0.056), ('ejemplo😂😂😂😂😂😂😂', 0.054)]
  [4/20] falso_positivo — @USER @USER Se sabía, pero dale q las gallinas mean  La izqu...


...

    Top tokens: [('.', 0.003), ('fac', 0.002), ('catal', -0.002), ('ha ', -0.001), ('Catalunya ', -0.001)]
  [16/20] falso_negativo — @USER ALGUIEN HA EXPKICADO PORQUE LOS ARGENTINOS NO QUIEREN ...


  0%|          | 0/498 [00:00<?, ?it/s]

    Top tokens: [('BA', 0.024), ('OS ', 0.015), ('N', 0.015), ('TIN', 0.015), ('USE', 0.014)]
  [17/20] falso_negativo — @USER No creo que muera antes de que los musulmanes tengan r...
    Top tokens: [('musulmanes ', 0.044), ('única ', 0.026), ('español', 0.011), ('es ', 0.01), ('representación ', -0.007)]
  [18/20] falso_negativo — lastima que no se volvio loco del todo y se suicido de paso....


  0%|          | 0/380 [00:00<?, ?it/s]

    Top tokens: [('loco ', 0.007), ('se ', 0.005), ('paso', -0.005), ('y ', 0.004), ('lastim', -0.003)]
  [19/20] falso_negativo — Un ladron menos q llega a España y ganamos un asesino mas!...
    Top tokens: [('ganamos ', -0.088), ('lad', 0.042), ('mas', 0.032), ('un ', 0.029), ('España ', 0.024)]
  [20/20] falso_negativo — Ahora pides asilo político HDP, ahora las escorias oficialis...
    Top tokens: [(', ', 0.038), ('esco', 0.033), ('rias ', 0.029), ('L', 0.023), ('istas ', 0.023)]

✅ Análisis SHAP guardado:
   CSV  → /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables/xai_analysis/shap_analysis_results.csv
   JSON → /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables/xai_analysis/shap_full_weights.json


In [22]:
# ── Celda: Paso 5.3 - ¿Los modismos aparecen entre los tokens relevantes? ──

import json

print("="*70)
print("  PASO 5.3 - MODISMOS EN TOKENS DE MAYOR PESO SHAP")
print("="*70)

df_shap = pd.read_csv(XAI_DIR / "shap_analysis_results.csv")

resultados = []

for _, row in df_shap.iterrows():
    top5 = json.loads(row["top5_tokens"])
    tiene_modismo = row["tiene_modismo"]
    error_type = row["error_type"]

    # Verificar si algún top-5 token es relevante (no padding, no subword)
    tokens_relevantes = [t for t in top5 if len(t) > 2 and not t.startswith("##")]

    resultados.append({
        "error_type":         error_type,
        "tiene_modismo":      tiene_modismo,
        "top5_tokens":        top5,
        "tokens_relevantes":  tokens_relevantes,
        "n_relevantes":       len(tokens_relevantes),
    })

df_res = pd.DataFrame(resultados)

print(f"\nResumen:")
print(f"  Textos CON modismo  : {df_res['tiene_modismo'].sum()}")
print(f"  Textos SIN modismo  : {(~df_res['tiene_modismo']).sum()}")

print(f"\nTokens relevantes promedio en top-5:")
print(df_res.groupby("error_type")["n_relevantes"].mean().round(2))

# Guardar
modismo_file = XAI_DIR / "shap_modismo_tokens.csv"
df_res.to_csv(modismo_file, index=False)
print(f"\n✅ Guardado → {modismo_file}")

  PASO 5.3 - MODISMOS EN TOKENS DE MAYOR PESO SHAP

Resumen:
  Textos CON modismo  : 12
  Textos SIN modismo  : 8

Tokens relevantes promedio en top-5:
error_type
falso_negativo    3.9
falso_positivo    4.0
Name: n_relevantes, dtype: float64

✅ Guardado → /content/drive/MyDrive/unmsm/ciclo 2026-1/tesis/COLAB/reports/tables/xai_analysis/shap_modismo_tokens.csv
